# rdf-augmenter - Quickstart

This notebook augments a tiny RDF knowledge graph in **under five seconds**, with no GPU and no model downloads, using the offline, deterministic **lexicon backend**.

You will:
1. Load an RDF graph (8 people, each with a name and an occupation).
2. Augment the `occupation` literals with synonyms, hypernyms (types) and related terms.
3. Inspect the new triples, export the result, and write a reproducibility manifest.

> **New to RDF?** Each fact is a *triple*: `(subject, predicate, object)`, e.g. `(ex:Person2, ex:occupation, "Surgeon")`. A knowledge graph is just a set of such triples.

In [1]:
# Make the package importable whether you pip-installed rdf-augmenter
# or are running these notebooks straight from the cloned repository.
import sys, pathlib
try:
    import rdf_augmenter  # noqa: F401
except ModuleNotFoundError:
    repo = pathlib.Path.cwd().parent
    sys.path.insert(0, str(repo))
    import rdf_augmenter  # noqa: F401
print('rdf-augmenter', rdf_augmenter.__version__)


rdf-augmenter 0.2.0


## 1. Load the sample graph

In [2]:
import pathlib
SAMPLE = next(p for p in [pathlib.Path('../examples/sample_people.ttl'),
                          pathlib.Path('examples/sample_people.ttl')] if p.exists())
from rdf_augmenter import RDFAugmenter

aug = RDFAugmenter(seed=42)          # seed -> reproducible runs
aug.load(str(SAMPLE))
print(f'Loaded {len(aug.graph)} triples')
print(aug.export(fmt='turtle')[:400])


Loaded 16 triples
@prefix ex: <http://example.org/> .

ex:Person1 ex:name "John" ;
    ex:occupation "Data Scientist" .

ex:Person2 ex:name "Alice" ;
    ex:occupation "Surgeon" .

ex:Person3 ex:name "Michael" ;
    ex:occupation "Architect" .

ex:Person4 ex:name "Sophie" ;
    ex:occupation "Journalist" .

ex:Person5 ex:name "Emma" ;
    ex:occupation "Lawyer" .

ex:Person6 ex:name "Noah" ;
    ex:occupation "Teac


## 2. Augment the occupation literals

We restrict augmentation to the `ex:occupation` predicate. The report tells us exactly how many triples were added, broken down by relation type.

In [3]:
report = aug.augment(
    predicates=['http://example.org/occupation'],
    relations=('synonym', 'hypernym', 'related'),
    top_k=4,
)
print(report.summary())


RDF augmentation report
-----------------------
backend            : lexicon
seed               : 42
triples before     : 16
triples after      : 105
triples added      : 78
  - hypernym      : 24
  - related       : 32
  - synonym       : 22


## 3. Look at some of the new triples

In [4]:
from rdflib import URIRef, Literal
person2 = URIRef('http://example.org/Person2')   # Alice, the Surgeon
for p, o in sorted(aug.graph.predicate_objects(person2), key=lambda x: str(x)):
    print(f'{aug.graph.namespace_manager.normalizeUri(p):<22} {o}')


ex:name                Alice
ex:occupation          Surgeon
skos:altLabel          operating physician
skos:altLabel          sawbones
aug:hypernym           doctor
aug:hypernym           medical practitioner
aug:hypernym           physician
aug:hypernym           professional
aug:relatedTerm        anatomy
aug:relatedTerm        hospital
aug:relatedTerm        operating room
aug:relatedTerm        surgery


## 4. Export and record provenance

The augmented graph can be exported in any standard RDF syntax. The manifest captures the seed, parameters, input hash and library versions so anyone can reproduce this exact run.

In [5]:
aug.export('augmented.ttl', fmt='turtle')
aug.export('augmented.jsonld', fmt='json-ld')
report.save_manifest('run_manifest.json')

import json
manifest = json.load(open('run_manifest.json'))
print(json.dumps(manifest['output'], indent=2))


{
  "triples": 105,
  "triples_added": 78,
  "added_by_relation": {
    "synonym": 22,
    "hypernym": 24,
    "related": 32
  },
  "concepts_created": 0,
  "stats": {
    "triples": 105,
    "subjects": 10,
    "predicates": 15,
    "objects": 94,
    "literals": 102,
    "nodes": 103,
    "density": 1.0194,
    "predicate_counts": {
      "http://example.org/name": 8,
      "http://example.org/occupation": 8,
      "http://www.w3.org/1999/02/22-rdf-syntax-ns#type": 2,
      "http://www.w3.org/2000/01/rdf-schema#label": 1,
      "http://www.w3.org/2004/02/skos/core#altLabel": 22,
      "http://www.w3.org/ns/prov#startedAtTime": 1,
      "http://www.w3.org/ns/prov#wasAssociatedWith": 1,
      "https://w3id.org/rdf-augmenter/ns#attach": 1,
      "https://w3id.org/rdf-augmenter/ns#backend": 1,
      "https://w3id.org/rdf-augmenter/ns#hypernym": 24,
      "https://w3id.org/rdf-augmenter/ns#ratio": 1,
      "https://w3id.org/rdf-augmenter/ns#relatedTerm": 32,
      "https://w3id.org/rdf-au

## Next steps

* Swap in a richer backend: `RDFAugmenter(backend=WordNetBackend())` or `BertBackend()` (optional installs).
* Build a SKOS thesaurus instead of subject-attached literals with `aug.augment(..., attach='concept')`.
* See **`02_visualizing_augmentation.ipynb`** to chart the augmentation effect.
* Read `docs/tutorials/` for the beginner and advanced guides.